# S6E10 — 16 Teacher models trained on the original survey

> **Pipeline position:** 03 / 09 captains (`orig_logit`, `og_cat_logit`) → **16 more teachers** → 17 RealMLP-rich + teachers → 11 stack

| | |
|---|---|
| **Input** | `data/raw/train.csv`, `test.csv`, `orig_data.csv` |
| **Output** | `data/processed/X_teach_train.parquet`, `X_teach_test.parquet` (`og_xgb_logit`, `og_realmlp_logit`) |
| **Runtime** | ≈ 3–5 min (GPU) |
| **Labels used?** | **Only the original survey's labels.** Competition labels are read once, to *report* the teachers' AUC on competition train, never for training. |

**Why:** the competition data was generated from the original survey. A model trained only on the original rows gives every competition row a "second opinion" that comes from *outside* the competition data — the only kind of information that moved the score in this project (`02`, `09`, `10`; the in-row ideas of `12`–`15` gave nothing). We already have two such teachers: the LightGBM captain `orig_logit` (03, AUC 0.95472) and the CatBoost captain `og_cat_logit` (09, 0.9544). This notebook adds two of a different kind, as used by the author of the public TabPFN members:
* **`og_xgb_logit`** — XGBoost, depth 8 (deeper trees than the captains: more interactions);
* **`og_realmlp_logit`** — RealMLP with the recipe of `07` (a network: a different view from the tree captains).

**How each teacher is trained:** 5-fold CV *on the original rows* (to report an honest AUC on the original data); the 5 fold models then all predict the competition rows and their logits are averaged. No refit is needed, and no competition row is ever used for training.

**Duplicates:** 21 competition rows (17 train, 4 test) are identical to an original row on all 21 columns. Those original rows are removed before training, so a teacher cannot simply recall the label of a row it will later score.

> **Glossary:** *Teacher / captain* = a model trained on the original survey whose prediction becomes a feature for the competition data. *Logit* = log(p / (1 − p)), the scale networks and the stacker work with.

In [1]:
import torch          # first import: on the Mac the kernel crashes if torch comes after numpy/sklearn (OpenMP conflict)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import re, time, gc, warnings
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import xgboost as xgb
from pytabkit import RealMLP_TD_Classifier
warnings.filterwarnings('ignore')

SMOKE = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RAW = next(p for p in [Path('data/raw'), Path('../data/raw')] if (p / 'train.csv').exists())
PROC = next(p for p in [Path('data/processed'), Path('../data/processed')] if (p / 'X_rich_train.parquet').exists())

train = pd.read_csv(RAW / 'train.csv'); test = pd.read_csv(RAW / 'test.csv'); orig = pd.read_csv(RAW / 'orig_data.csv')
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [c for c in test.columns if c not in CATS + NUMS + ['id']]
BASE = NUMS + RATINGS + CATS
assert len(BASE) == 21 and all(c in orig.columns for c in BASE)
tobin = lambda s: s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'satisfied']).astype(int)

# Remove original rows that are exact copies (all 21 columns) of a competition row.
key = lambda d: d[BASE].astype(str).agg('|'.join, axis=1)
comp_keys = set(key(train)) | set(key(test))
dup = key(orig).isin(comp_keys)
og = orig.loc[~dup].reset_index(drop=True)
y_og = tobin(og['satisfaction']).values
print(f'original rows {len(orig):,} -> {len(og):,} after removing {dup.sum()} exact copies of competition rows | satisfied share {y_og.mean():.3f}')

full = pd.concat([train[BASE], test[BASE]], ignore_index=True); n_train = len(train)
y_comp = tobin(train['satisfaction']).values          # used ONLY to report AUC below, never for training

C:\Users\89671\anaconda3\envs\kaggle-playground\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


W1010 04:52:36.813000 25204 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


original rows 129,880 -> 129,859 after removing 21 exact copies of competition rows | satisfied share 0.434


## 1. Common encoding
Categoricals → the same integer codes for original and competition rows (vocabulary from train+test, as in `09`). XGBoost handles the missing arrival delays natively; for RealMLP they are filled with the departure delay plus a missing flag (the rule of `03`).

In [2]:
CAT_MAPS = {c: {v: i for i, v in enumerate(sorted(pd.concat([train[c], test[c]]).dropna().unique()))} for c in CATS}
def enc(df, fill=False):
    X = df[BASE].copy()
    for c in CATS: X[c] = X[c].map(CAT_MAPS[c]).fillna(-1).astype('int16')
    if fill:
        X['arr_missing'] = X['Arrival Delay in Minutes'].isna().astype('int8')
        X['Arrival Delay in Minutes'] = X['Arrival Delay in Minutes'].fillna(X['Departure Delay in Minutes'])
    X.columns = [re.sub(r'[^0-9a-zA-Z]+', '_', c).strip('_').lower() for c in X.columns]
    return X
X_og, X_full = enc(og), enc(full)
X_og_f, X_full_f = enc(og, fill=True), enc(full, fill=True)
OG_FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X_og, y_og))
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
print(X_og.shape, X_full.shape)

(129859, 21) (999479, 21)


## 2. Teacher 1 — XGBoost, depth 8
Early stopping on the held-out part of the *original* data picks the number of trees per fold.

In [3]:
t0 = time.time(); oof_og = np.zeros(len(og)); pred = np.zeros(len(full)); best_its = []
for a, b in OG_FOLDS:
    m = xgb.XGBClassifier(max_depth=8, learning_rate=0.03, n_estimators=50 if SMOKE else 3000, subsample=0.8, colsample_bytree=0.8,
                          min_child_weight=3, reg_lambda=1.0, tree_method='hist', device=DEVICE, eval_metric='auc',
                          early_stopping_rounds=100, random_state=42, verbosity=0)
    m.fit(X_og.iloc[a], y_og[a], eval_set=[(X_og.iloc[b], y_og[b])], verbose=False)
    oof_og[b] = m.predict_proba(X_og.iloc[b])[:, 1]; best_its.append(m.best_iteration)
    pred += lg(m.predict_proba(X_full)[:, 1]) / len(OG_FOLDS)     # average of the 5 fold models, on the logit scale
og_xgb_logit = pred.astype('float32')
print(f'XGB d8: AUC on original (5-fold OOF) {roc_auc_score(y_og, oof_og):.5f} | trees per fold {best_its} | '
      f'AUC on competition train {roc_auc_score(y_comp, og_xgb_logit[:n_train]):.5f} | {time.time()-t0:.0f}s')

XGB d8: AUC on original (5-fold OOF) 0.99498 | trees per fold [701, 522, 684, 641, 665] | AUC on competition train 0.95491 | 16s


## 3. Teacher 2 — RealMLP (recipe of `07`)
Categorical twins for route, age and the 13 ratings; 4 networks, 4 epochs, no early stopping (the schedule regularises).

In [4]:
RAT = [re.sub(r'[^0-9a-zA-Z]+', '_', c).strip('_').lower() for c in RATINGS]
TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RAT}}
TW_LEVELS = {t: sorted(pd.concat([X_og_f[s], X_full_f[s]]).astype(int).unique()) for t, s in TWINS.items()}
CATC = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(TWINS)
def nnf(D):
    D = D.copy()
    for c in ['gender', 'customer_type', 'type_of_travel', 'class']: D[c] = D[c].astype('category')
    for t, s in TWINS.items(): D[t] = pd.Categorical(D[s].astype(int), categories=TW_LEVELS[t])
    return D
PARAMS = dict(n_epochs=1 if SMOKE else 4, n_ens=4, batch_size=256, use_early_stopping=False, device=DEVICE, verbosity=0,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='lin_cos_log_15', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33,
    use_ls=True, ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False, bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'])
t0 = time.time(); oof_og2 = np.zeros(len(og)); pred = np.zeros(len(full)); XF = nnf(X_full_f); XO = nnf(X_og_f)
for i, (a, b) in enumerate(OG_FOLDS):
    m = RealMLP_TD_Classifier(**PARAMS, random_state=i)
    m.fit(XO.iloc[a], y_og[a], X_val=XO.iloc[b], y_val=y_og[b], cat_col_names=CATC)   # X_val = original rows only (logging; no early stopping)
    oof_og2[b] = m.predict_proba(XO.iloc[b])[:, 1]
    pred += lg(m.predict_proba(XF)[:, 1]) / len(OG_FOLDS)
    del m; gc.collect(); torch.cuda.empty_cache() if DEVICE == 'cuda' else None
og_realmlp_logit = pred.astype('float32')
print(f'RealMLP: AUC on original (5-fold OOF) {roc_auc_score(y_og, oof_og2):.5f} | '
      f'AUC on competition train {roc_auc_score(y_comp, og_realmlp_logit[:n_train]):.5f} | {time.time()-t0:.0f}s')

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


RealMLP: AUC on original (5-fold OOF) 0.99500 | AUC on competition train 0.95291 | 105s


## 4. Compare with the existing captains and save

In [5]:
Xr = pd.read_parquet(PROC / 'X_rich_train.parquet', columns=['orig_logit', 'og_cat_logit'])
T = pd.DataFrame({'og_xgb_logit': og_xgb_logit[:n_train], 'og_realmlp_logit': og_realmlp_logit[:n_train],
                  'orig_logit (03)': Xr['orig_logit'].values, 'og_cat_logit (09)': Xr['og_cat_logit'].values})
print('AUC on competition train:', {c: round(roc_auc_score(y_comp, T[c]), 5) for c in T})
print('Spearman correlation:'); print(T.corr(method='spearman').round(4).to_string())
if not SMOKE:
    out = pd.DataFrame({'og_xgb_logit': og_xgb_logit, 'og_realmlp_logit': og_realmlp_logit})
    assert not out.isna().any().any()
    out.iloc[:n_train].reset_index(drop=True).to_parquet(PROC / 'X_teach_train.parquet')
    out.iloc[n_train:].reset_index(drop=True).to_parquet(PROC / 'X_teach_test.parquet')
    print('saved X_teach_*', out.shape)

AUC on competition train: {'og_xgb_logit': 0.95491, 'og_realmlp_logit': 0.95291, 'orig_logit (03)': 0.95472, 'og_cat_logit (09)': 0.95436}
Spearman correlation:


                   og_xgb_logit  og_realmlp_logit  orig_logit (03)  og_cat_logit (09)
og_xgb_logit             1.0000            0.9698           0.9942             0.9799
og_realmlp_logit         0.9698            1.0000           0.9691             0.9600
orig_logit (03)          0.9942            0.9691           1.0000             0.9789
og_cat_logit (09)        0.9799            0.9600           0.9789             1.0000
saved X_teach_* (999479, 2)


## Results & decisions

| Teacher | AUC on original (5-fold OOF) | AUC on competition train | Spearman with `orig_logit` (03) | Time |
|---|---|---|---|---|
| `og_xgb_logit` (XGBoost d8) | 0.99498 | 0.95491 | 0.994 | 16 s (GPU) |
| `og_realmlp_logit` (RealMLP) | 0.99500 | 0.95291 | 0.969 | 105 s (GPU) |
| *`orig_logit` (03, LightGBM)* | | *0.95472* | *1* | |
| *`og_cat_logit` (09, CatBoost)* | | *0.95436* | *0.979* | |

21 original rows identical to competition rows were removed before training. The XGBoost teacher is almost a copy of the LightGBM captain; the RealMLP teacher is more different but weaker on the competition data. The original survey is far easier to predict (AUC 0.995) than its synthetic copy (≈ 0.955): the generator added noise that no teacher can recover.

**Usefulness (decided in `17`): none** — see there. The features stay on disk for reproducibility and are not used downstream.


## Glossary
* **Teacher** — a model trained only on the original survey; its prediction for a competition row is used as a feature.
* **Exact copy** — a competition row identical to an original row on all 21 columns (21 such rows).
* **Fold-model averaging** — the 5 CV models of the teacher all predict the competition rows; their logits are averaged instead of refitting once more.